In [1]:
import os
import glob
import sys

import pandas as pd
import numpy as np
from pathlib import Path

import random
from tqdm import tqdm
import pydicom  # Handle MRI images

import cv2  # OpenCV

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Fix: TF 2.18 + current Kaggle protobuf runtime can raise
# AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
# For this environment, forcing the pure-Python protobuf backend avoids that crash.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ.pop("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", None)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical

np.random.seed(0)
random.seed(12)
tf.random.set_seed(12)

print("TF:", tf.__version__)
print("pydicom:", pydicom.__version__)



2026-01-23 11:45:10.032738: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769168710.047197      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769168710.051550      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF: 2.18.0
pydicom: 3.0.1


In [2]:
data_dir = Path("../input/rsna-miccai-brain-tumor-radiogenomic-classification/")

mri_types = ["FLAIR", "T1w", "T2w", "T1wCE"]
excluded_images = [109, 123, 709]  # Bad images



In [3]:
train_df = pd.read_csv(data_dir / "train_labels.csv")
test_df = pd.read_csv(data_dir / "sample_submission.csv")
sample_submission = pd.read_csv(data_dir / "sample_submission.csv")

train_df["BraTS21ID"] = train_df["BraTS21ID"].astype(int)
test_df["BraTS21ID"] = test_df["BraTS21ID"].astype(int)
sample_submission["BraTS21ID"] = sample_submission["BraTS21ID"].astype(int)

train_df = train_df[~train_df.BraTS21ID.isin(excluded_images)].reset_index(drop=True)

print(f"train data: Rows={train_df.shape[0]}, Columns={train_df.shape[1]}")
print(f"test data: Rows={test_df.shape[0]}, Columns={test_df.shape[1]}")




train data: Rows=523, Columns=2
test data: Rows=59, Columns=2


In [4]:
def load_dicom(path, size=512):
    """Read a DICOM image and return a resized uint8 grayscale image."""
    dicom = pydicom.dcmread(path)
    data = dicom.pixel_array.astype(np.float32)

    mx = float(np.max(data)) if data.size else 0.0
    if mx > 0:
        data = data / mx

    data = (data * 255.0).astype(np.uint8)
    return cv2.resize(data, (size, size), interpolation=cv2.INTER_AREA)




In [5]:
_PATH_CACHE = {}


def get_all_image_paths(brats21id, image_type, folder="train"):
    """Return a numpy array of selected slice paths for a patient and sequence."""
    assert image_type in mri_types

    key = (int(brats21id), image_type, folder)
    cached = _PATH_CACHE.get(key)
    if cached is not None:
        return cached

    patient_path = os.path.join(
        "../input/rsna-miccai-brain-tumor-radiogenomic-classification/%s/" % folder,
        str(brats21id).zfill(5),
    )

    paths = sorted(
        glob.glob(os.path.join(patient_path, image_type, "*")),
        key=lambda x: int(x[:-4].split("-")[-1]),
    )

    num_images = len(paths)
    start = int(num_images * 0.25)
    end = int(num_images * 0.75)

    interval = 3
    if num_images < 10:
        interval = 1

    out = np.array(paths[start:end:interval])
    _PATH_CACHE[key] = out
    return out


def get_all_images(brats21id, image_type, folder="train", size=225):
    return [
        load_dicom(path, size)
        for path in get_all_image_paths(brats21id, image_type, folder)
    ]




In [6]:
def get_all_data_for_train(image_type, image_size=430):
    global train_df

    ids = train_df["BraTS21ID"].to_numpy(dtype=np.int32)
    labels = train_df["MGMT_value"].to_numpy(dtype=np.int8)

    paths_per_id = []
    total_slices = 0
    for pid in ids:
        p = get_all_image_paths(int(pid), image_type, folder="train")
        paths_per_id.append(p)
        total_slices += len(p)

    X = np.empty((total_slices, image_size, image_size), dtype=np.uint8)
    y = np.empty((total_slices,), dtype=np.int8)
    train_ids = np.empty((total_slices,), dtype=np.int32)

    k = 0
    bad_reads = 0
    for pid, lab, paths in tqdm(list(zip(ids, labels, paths_per_id)), total=len(ids)):
        n = len(paths)
        if n == 0:
            continue

        n_written = 0
        for j, path in enumerate(paths):
            try:
                X[k + n_written] = load_dicom(path, image_size)
                n_written += 1
            except Exception:
                bad_reads += 1
                continue

        if n_written == 0:
            continue

        y[k : k + n_written] = lab
        train_ids[k : k + n_written] = pid
        k += n_written

    if k != total_slices:
        X = X[:k]
        y = y[:k]
        train_ids = train_ids[:k]

    print(f"Skipped unreadable train slices: {bad_reads}")
    return X, y, train_ids




In [7]:
def get_all_data_for_test(image_type, image_size=412):
    global test_df

    ids = test_df["BraTS21ID"].to_numpy(dtype=np.int32)

    paths_per_id = []
    total_slices = 0
    for pid in ids:
        p = get_all_image_paths(int(pid), image_type, folder="test")
        paths_per_id.append(p)
        total_slices += len(p)

    X = np.empty((total_slices, image_size, image_size), dtype=np.uint8)
    test_ids = np.empty((total_slices,), dtype=np.int32)

    k = 0
    bad_reads = 0
    for pid, paths in tqdm(list(zip(ids, paths_per_id)), total=len(ids)):
        n = len(paths)
        if n == 0:
            continue

        n_written = 0
        for j, path in enumerate(paths):
            try:
                X[k + n_written] = load_dicom(path, image_size)
                test_ids[k + n_written] = pid
                n_written += 1
            except Exception:
                bad_reads += 1
                continue

        k += n_written

    if k != total_slices:
        X = X[:k]
        test_ids = test_ids[:k]

    print(f"Skipped unreadable test slices: {bad_reads}")
    return X, test_ids




In [8]:
X, y, trainidt = get_all_data_for_train("T1wCE", image_size=32)
X_test, testidt = get_all_data_for_test("T1wCE", image_size=32)

print("Train arrays:", X.shape, y.shape, trainidt.shape)
print("Test arrays:", X_test.shape, testidt.shape)

assert X.shape[0] == y.shape[0] == trainidt.shape[0], "Train arrays misaligned"
assert X_test.shape[0] == testidt.shape[0], "Test arrays misaligned"
assert X.shape[0] > 0, "No training slices loaded; cannot train."
assert X_test.shape[0] > 0, "No test slices loaded; cannot predict."



100%|██████████| 523/523 [00:14<00:00, 35.08it/s]


Skipped unreadable train slices: 0


100%|██████████| 59/59 [00:01<00:00, 47.67it/s]

Skipped unreadable test slices: 0
Train arrays: (14563, 32, 32) (14563,) (14563,)
Test arrays: (1633, 32, 32) (1633,)


In [9]:
X_train, X_valid, y_train, y_valid, trainidt_train, trainidt_valid = train_test_split(
    X, y, trainidt, random_state=12
)



In [10]:
X_train = tf.expand_dims(X_train, axis=-1)
X_valid = tf.expand_dims(X_valid, axis=-1)
X_test = tf.expand_dims(X_test, axis=-1)

print("X_train:", X_train.shape, "X_valid:", X_valid.shape, "X_test:", X_test.shape)



I0000 00:00:1769168733.094926      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:45:00.0, compute capability: 8.6


X_train: (10922, 32, 32, 1) X_valid: (3641, 32, 32, 1) X_test: (1633, 32, 32, 1)


In [11]:
y_train = to_categorical(y_train, num_classes=2)
y_valid = to_categorical(y_valid, num_classes=2)

print("y_train:", y_train.shape, "y_valid:", y_valid.shape)




y_train: (10922, 2) y_valid: (3641, 2)


In [12]:
def get_model01(width=128, height=128, depth=64, name="3dcnn"):
    """Build a 3D convolutional neural network model."""

    inputs = tf.keras.Input((width, height, depth, 1))

    x = tf.keras.layers.Conv3D(filters=64, kernel_size=3, activation="relu")(inputs)
    x = tf.keras.layers.MaxPool3D(pool_size=2)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Conv3D(filters=64, kernel_size=3, activation="relu")(x)
    x = tf.keras.layers.MaxPool3D(pool_size=2)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Conv3D(filters=128, kernel_size=3, activation="relu")(x)
    x = tf.keras.layers.MaxPool3D(pool_size=2)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Conv3D(filters=256, kernel_size=3, activation="relu")(x)
    x = tf.keras.layers.MaxPool3D(pool_size=2)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.GlobalAveragePooling3D()(x)
    x = tf.keras.layers.Dense(units=512, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    outputs = tf.keras.layers.Dense(units=1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs, name=name)

    initial_learning_rate = 0.00009
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate, decay_steps=100000, decay_rate=0.96, staircase=True
    )
    model.compile(
        loss="binary_crossentropy",
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        metrics=["acc"],
    )

    return model




In [13]:
def get_model02():
    np.random.seed(0)
    random.seed(12)
    tf.random.set_seed(12)

    inpt = keras.Input(shape=X_train.shape[1:])

    h = keras.layers.Rescaling(1.0 / 255)(inpt)

    h = keras.layers.Conv2D(64, kernel_size=(4, 4), activation="relu", name="Conv_1")(h)
    h = keras.layers.MaxPool2D(pool_size=(2, 2))(h)

    h = keras.layers.Conv2D(32, kernel_size=(2, 2), activation="relu", name="Conv_2")(h)
    h = keras.layers.MaxPool2D(pool_size=(1, 1))(h)

    h = keras.layers.Dropout(0.1)(h)

    h = keras.layers.Flatten()(h)
    h = keras.layers.Dense(32, activation="relu")(h)

    output = keras.layers.Dense(2, activation="softmax")(h)

    model = keras.Model(inpt, output)

    model.compile(
        loss="categorical_crossentropy",
        optimizer="adam",
        metrics=[tf.keras.metrics.AUC(name="auc")],
    )
    return model




In [14]:
checkpoint_filepath = "best_model.keras"

model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=False,
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    save_freq="epoch",
    verbose=1,
)



In [15]:
BATCH_SIZE = 256

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .batch(BATCH_SIZE)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)
valid_ds = (
    tf.data.Dataset.from_tensor_slices((X_valid, y_valid))
    .batch(BATCH_SIZE)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

model = get_model02()
history = model.fit(
    train_ds,
    epochs=25,
    callbacks=[model_checkpoint_callback],
    validation_data=valid_ds,
    verbose=2,
)



Epoch 1/25


I0000 00:00:1769168734.830932      65 service.cc:148] XLA service 0x7fc83c00b640 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769168734.830969      65 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1769168734.958159      65 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1769168737.137923      65 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



Epoch 1: val_auc improved from -inf to 0.60248, saving model to best_model.keras
43/43 - 7s - 162ms/step - auc: 0.5793 - loss: 0.6840 - val_auc: 0.6025 - val_loss: 0.6767
Epoch 2/25

Epoch 2: val_auc improved from 0.60248 to 0.62026, saving model to best_model.keras
43/43 - 0s - 4ms/step - auc: 0.6053 - loss: 0.6762 - val_auc: 0.6203 - val_loss: 0.6711
Epoch 3/25

Epoch 3: val_auc improved from 0.62026 to 0.63606, saving model to best_model.keras
43/43 - 0s - 4ms/step - auc: 0.6285 - loss: 0.6676 - val_auc: 0.6361 - val_loss: 0.6658
Epoch 4/25

Epoch 4: val_auc improved from 0.63606 to 0.65152, saving model to best_model.keras
43/43 - 0s - 4ms/step - auc: 0.6467 - loss: 0.6594 - val_auc: 0.6515 - val_loss: 0.6591
Epoch 5/25

Epoch 5: val_auc improved from 0.65152 to 0.65656, saving model to best_model.keras
43/43 - 0s - 4ms/step - auc: 0.6716 - loss: 0.6465 - val_auc: 0.6566 - val_loss: 0.6541
Epoch 6/25

Epoch 6: val_auc improved from 0.65656 to 0.66813, saving model to best_model.ke

In [16]:
model_best = tf.keras.models.load_model(filepath=checkpoint_filepath)



In [17]:
valid_pred_ds = (
    tf.data.Dataset.from_tensor_slices(X_valid)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
y_pred = model_best.predict(valid_pred_ds, verbose=0)
pred_prob = y_pred[:, 1]  # probability of MGMT_value == 1

result = pd.DataFrame(
    {"BraTS21ID": trainidt_valid.astype(int), "MGMT_value": pred_prob}
)
result2 = result.groupby("BraTS21ID", as_index=False).mean()

result2 = result2.merge(
    train_df, on="BraTS21ID", how="inner", suffixes=("_pred", "_true")
)
auc = roc_auc_score(result2["MGMT_value_true"], result2["MGMT_value_pred"])
print(f"Validation AUC={auc}")



Validation AUC=0.8909722222222222


In [18]:
try:
    test_pred_ds = (
        tf.data.Dataset.from_tensor_slices(X_test)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )
    y_pred_test = model_best.predict(test_pred_ds, verbose=0)
    pred_prob_test = y_pred_test[:, 1]

    result_test = pd.DataFrame(
        {"BraTS21ID": testidt.astype(int), "MGMT_value": pred_prob_test}
    )

    sub = result_test.groupby("BraTS21ID", as_index=False).mean()
    sub = sample_submission[["BraTS21ID"]].merge(sub, on="BraTS21ID", how="left")
    sub["MGMT_value"] = sub["MGMT_value"].fillna(0.5).astype(float)

    sub = sub[["BraTS21ID", "MGMT_value"]]
except Exception as e:
    print("Fallback submission due to error during test prediction:", repr(e))
    sub = sample_submission.copy()
    sub["MGMT_value"] = 0.5

sub.to_csv("submission.csv", index=False)

print(sub.head())
print("Wrote submission.csv with shape:", sub.shape)
print("submission.csv columns:", list(sub.columns))
assert list(sub.columns) == ["BraTS21ID", "MGMT_value"]
assert sub.shape[0] == sample_submission.shape[0]

   BraTS21ID  MGMT_value
0        356    0.249102
1        140    0.609081
2        757    0.615162
3        275    0.588477
4        570    0.571741
Wrote submission.csv with shape: (59, 2)
submission.csv columns: ['BraTS21ID', 'MGMT_value']
